# Vivaah — WAN 2.1 Image-to-Video (lehenga hero clips)

Generates the slowly-rotating garment clip from a prepped image, on a **free Colab T4**.

**Runtime → Change runtime type → T4 GPU** before you start.

**Folders (in your Google Drive):**
- `vivaah/raw` — original lehenga photos (upload here)
- `vivaah/prepped` — background removed + light studio backdrop (this notebook writes here)
- `vivaah/videos` — finished MP4s (this notebook writes here)

Order: **1) GPU check → 2) install → 3) mount Drive → 4) config → 5) prep (bg removal) → 6) load WAN → 7) generate.**

> ⚠️ **Free T4 is slow:** expect roughly **20–40 min per 5-second clip**. bf16 is used because it is
> numerically safe on the T4 (fp16 can give black/NaN frames). For speed use a Colab A100, or the
> GGUF/ComfyUI route. Do a few clips per session and let it batch the rest across sessions.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies (~2–3 min)

In [ ]:
!pip install -q git+https://github.com/huggingface/diffusers
!pip install -q transformers accelerate ftfy safetensors sentencepiece \
  imageio imageio-ffmpeg rembg onnxruntime Pillow

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Config — edit paths/prompt here, then run every session

In [ ]:
RAW_DIR     = "/content/drive/MyDrive/vivaah/raw"       # your original photos
PREPPED_DIR = "/content/drive/MyDrive/vivaah/prepped"   # bg-removed + light studio backdrop
OUTPUT_DIR  = "/content/drive/MyDrive/vivaah/videos"    # finished MP4s

WIDTH, HEIGHT = 832, 480    # WAN 480P 16:9 landscape (both divisible by 16)
NUM_FRAMES    = 81          # 81 frames @ 16fps = ~5s  (use 49 for faster)
FPS           = 16
STEPS         = 30
GUIDANCE      = 5.5
SEED          = 42          # keep the same seed across all lehengas for a consistent set

PROMPT = (
    "A sage-green net bridal lehenga displayed on a faceless ivory mannequin, floating centered in the frame. Fitted sleeveless blouse with delicate silver-grey floral thread embroidery, sequins and pearl beadwork; long floor-length flared skirt in soft translucent net with dense pastel floral embroidery, scattered sequin flowers and a scalloped silver-embroidered hem; a sheer mint-green dupatta with an embroidered floral border draped over one shoulder; a fine silver choker necklace. The mannequin slowly and smoothly rotates around its vertical axis on an invisible turntable, gradually revealing the front, side and back of the outfit. The sheer net fabric and dupatta sway gently and catch the light; sequins and embroidery softly shimmer as they move. Static locked-off camera at eye level, centered composition, shallow depth of field. Clean soft light-grey studio background, bright and airy, with a gentle overhead spotlight and a soft contact shadow on the floor beneath the garment. Smooth elegant slow motion, premium e-commerce product look. Photorealistic, ultra-detailed, 4K, smooth stable motion, no flicker."
)

NEGATIVE_PROMPT = (
    "distorted fabric, morphing embroidery, changing patterns, warping textures, melting, glitch, jitter, flicker, stutter, extra limbs, human face, moving hands, deformed mannequin, duplicate dupatta, extra clothing, floating debris, blurry, low quality, low resolution, oversaturated, harsh lighting, watermark, text, logo, caption, cluttered background, colored background, props, second outfit"
)

import os
for d in (RAW_DIR, PREPPED_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)
print("Config loaded. Put photos in:", RAW_DIR)

## 5. Prep: remove background + light studio composite

Same result as `tools/prep_lehenga.py` (studio style): a bright light-grey backdrop with a soft
spotlight and a floor shadow, like the e-commerce reference. Runs here so you need no local install.
Reads every image in `RAW_DIR`, writes clean PNGs to `PREPPED_DIR`.

> Note: this keeps the mannequin (background removal can't turn it into a ghost-mannequin).
Skip this cell if you already prepped your images locally.

In [ ]:
from PIL import Image, ImageDraw, ImageFilter
from rembg import remove, new_session
import glob, os

# light studio palette (matches the e-commerce reference)
TOP, BOTTOM, HILITE = (233,232,230), (214,212,210), (246,245,243)
FILL = 0.86
_session = new_session("isnet-general-use")

def _gradient(w, h):
    col = Image.new("RGB", (1, h))
    for y in range(h):
        t = y / max(h-1, 1)
        col.putpixel((0, y), tuple(int(a+(b-a)*t) for a,b in zip(TOP, BOTTOM)))
    return col.resize((w, h))

def _spotlight(bg, strength=0.62, cy=0.40):
    w, h = bg.size
    m = Image.new("L", (w, h), 0)
    d = ImageDraw.Draw(m)
    cx, ccy, ex, ey = w//2, int(h*cy), int(w*0.78), int(h*0.62)
    d.ellipse([cx-ex, ccy-ey, cx+ex, ccy+ey], fill=int(255*strength))
    m = m.filter(ImageFilter.GaussianBlur(max(w, h)//5))
    return Image.composite(Image.new("RGB", (w, h), HILITE), bg, m)

def _shadow(size, cut, pos, opacity=100):
    w, h = size; x, y = pos
    m = Image.new("L", (w, h), 0); d = ImageDraw.Draw(m)
    gx = x + cut.width//2; gy = y + cut.height - int(cut.height*0.015)
    ew, eh = int(cut.width*0.52), int(cut.width*0.09)
    d.ellipse([gx-ew, gy-eh, gx+ew, gy+eh], fill=opacity)
    m = m.filter(ImageFilter.GaussianBlur(int(cut.width*0.06)))
    sh = Image.new("RGBA", (w, h), (45,43,43,255)); sh.putalpha(m); return sh

def prep(path, out_path):
    img = Image.open(path).convert("RGBA")
    cut = remove(img, session=_session, alpha_matting=True,
                 alpha_matting_foreground_threshold=245,
                 alpha_matting_background_threshold=15, alpha_matting_erode_size=8)
    bbox = cut.split()[-1].getbbox()
    if bbox: cut = cut.crop(bbox)
    scale = int(HEIGHT*FILL) / cut.height
    if cut.width*scale > WIDTH*0.94: scale = (WIDTH*0.94) / cut.width
    cut = cut.resize((max(1,int(cut.width*scale)), max(1,int(cut.height*scale))))
    x = (WIDTH - cut.width)//2
    y = max(int(HEIGHT*0.03), int(HEIGHT*0.90) - cut.height)
    canvas = _spotlight(_gradient(WIDTH, HEIGHT)).convert("RGBA")
    canvas = Image.alpha_composite(canvas, _shadow((WIDTH,HEIGHT), cut, (x,y)))
    canvas.paste(cut, (x, y), cut)
    canvas.convert("RGB").save(out_path, "PNG")

exts = ("*.jpg","*.jpeg","*.png","*.webp")
raw = sorted(f for e in exts for f in glob.glob(os.path.join(RAW_DIR, e)))
print(f"Prepping {len(raw)} image(s)...")
for i, p in enumerate(raw, 1):
    out = os.path.join(PREPPED_DIR, os.path.splitext(os.path.basename(p))[0] + ".png")
    prep(p, out)
    print(f"  [{i}/{len(raw)}] {os.path.basename(p)} -> {os.path.basename(out)}")
print("Prep done ->", PREPPED_DIR)

## 6. Load WAN 2.1 I2V (first run downloads the model — a few minutes)

In [ ]:
import torch
from diffusers import WanImageToVideoPipeline, AutoencoderKLWan
from transformers import CLIPVisionModel

MODEL_ID = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"

image_encoder = CLIPVisionModel.from_pretrained(MODEL_ID, subfolder="image_encoder", torch_dtype=torch.float32)
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(
    MODEL_ID, vae=vae, image_encoder=image_encoder, torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()   # fits the 14B model on a 16GB T4
pipe.vae.enable_tiling()
print("WAN 2.1 I2V ready.")

## 7. Batch generate — one MP4 per prepped image

Runs every PNG in `PREPPED_DIR`. Already-generated clips are skipped, so you can
stop and resume across sessions until all ~30 are done.

In [ ]:
from diffusers.utils import export_to_video, load_image
import glob, os, time

gen = torch.Generator(device="cuda").manual_seed(SEED)
imgs = sorted(glob.glob(os.path.join(PREPPED_DIR, "*.png")) +
              glob.glob(os.path.join(PREPPED_DIR, "*.jpg")))
print(f"{len(imgs)} prepped image(s) to render.")

for i, p in enumerate(imgs, 1):
    name = os.path.splitext(os.path.basename(p))[0] + ".mp4"
    dst = os.path.join(OUTPUT_DIR, name)
    if os.path.exists(dst):
        print(f"  [{i}/{len(imgs)}] {name} already done, skipping.")
        continue
    t0 = time.time()
    image = load_image(p).resize((WIDTH, HEIGHT))
    frames = pipe(
        image=image, prompt=PROMPT, negative_prompt=NEGATIVE_PROMPT,
        height=HEIGHT, width=WIDTH, num_frames=NUM_FRAMES,
        num_inference_steps=STEPS, guidance_scale=GUIDANCE, generator=gen,
    ).frames[0]
    export_to_video(frames, dst, fps=FPS)
    print(f"  [{i}/{len(imgs)}] {name}  ({time.time()-t0:.0f}s) -> {dst}")
print("All done ->", OUTPUT_DIR)